In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Esto es para obtener el código para el estado de Oaxaca

In [2]:
f = '../data/raw/catun_municipio/AGEEML_20258131423976_utf.csv'
ubica_geo = pd.read_csv(f, encoding="utf-8"
                       # usecols = ['folioviv','aire_acond','entidad','factor']
                      )
ubica_geo

,MAPA,CVE_ENT,NOM_ENT,NOM_ABR,CVE_MUN,NOM_MUN,CVE_CAB,NOM_CAB,POB_TOTAL,POB_MASCULINA,POB_FEMENINA,TOTAL DE VIVIENDAS HABITADAS
0,1001,1,Aguascalientes,Ags.,1,Aguascalientes,0001,Aguascalientes,948990,462073,486917,266942
1,1002,1,Aguascalientes,Ags.,2,Asientos,0001,Asientos,51536,25261,26275,12544
2,1003,1,Aguascalientes,Ags.,3,Calvillo,0001,Calvillo,58250,28563,29687,15556
3,1004,1,Aguascalientes,Ags.,4,Cosío,0001,Cosío,17000,8292,8708,3938
4,1005,1,Aguascalientes,Ags.,5,Jesús María,0001,Jesús María,129929,64219,65710,33229
...,...,...,...,...,...,...,...,...,...,...,...,...
2473,32054,32,Zacatecas,Zac.,54,Villa Hidalgo,0001,Villa Hidalgo,19446,9504,9942,4951
2474,32055,32,Zacatecas,Zac.,55,Villanueva,0001,Villanueva,31558,15590,15968,9052
2475,32056,32,Zacatecas,Zac.,56,Zacatecas,0001,Zacatecas,149607,71972,77635,42424
2476,32057,32,Zacatecas,Zac.,57,Trancoso,0001,Trancoso,20455,10039,10416,4671


In [3]:
ubica_geo.dtypes

MAPA                             int64
CVE_ENT                          int64
NOM_ENT                         object
NOM_ABR                         object
CVE_MUN                          int64
NOM_MUN                         object
CVE_CAB                         object
NOM_CAB                         object
POB_TOTAL                       object
POB_MASCULINA                   object
POB_FEMENINA                    object
TOTAL DE VIVIENDAS HABITADAS    object
dtype: object

## Importar y acotar tabla VIVIENDA para Oaxaca

In [4]:
f = '../data/raw/enigh2024_ns_viviendas_csv/viviendas.csv'
VIVIENDA = pd.read_csv(f, 
                       usecols = ['folioviv','ubica_geo'
                                  # 'factor',
                                  # 'mat_pared','mat_techos','mat_pisos',
                                  # 'cuart_dorm','num_cuarto',
                                  # # no se incluyeron banos
                                  # 'disp_elect',
                                  # 'tot_resid','est_socio',
                                  # # 'tot_hog',
                                  # # Abajo son cargas electricas
                                  # 'aire_acond','calefacc',
                                  # 'focos','focos_ahor', 
                                  # 'combus'
                                 ] #Es el combustible para cocinar, si es electricidad (5) va a representar una carga electrica importante
                      )
VIVIENDA

,folioviv,ubica_geo
0,100001901,1001
1,100001902,1001
2,100001904,1001
3,100001905,1001
4,100002501,1001
...,...,...
90319,3260593814,32052
90320,3260593815,32052
90321,3260593816,32052
90322,3260593817,32052


In [5]:
# Asegurar que la columna es de texto
VIVIENDA['ubica_geo'] = VIVIENDA['ubica_geo'].astype(str)

# Crear las nuevas columnas
VIVIENDA['munic'] = VIVIENDA['ubica_geo'].str[-3:]   # últimos 3 caracteres
VIVIENDA['edo']   = VIVIENDA['ubica_geo'].str[:-3]   # el resto (al inicio)
VIVIENDA = VIVIENDA.astype(int)


In [6]:
VIVIENDA.dtypes

folioviv     int64
ubica_geo    int64
munic        int64
edo          int64
dtype: object

In [7]:
resumen_munic = (
    VIVIENDA
    .groupby('edo')['munic']
    .nunique()
    .reset_index()
    .rename(columns={'munic': 'munic_ENIGH'})
    .sort_values(by='edo', ascending=True)  # orden ascendente por edo
    .reset_index(drop=True)  # opcional: limpia el índice
)

resumen_munic

,edo,munic_ENIGH
0,1,10
1,2,6
2,3,5
3,4,11
4,5,29
5,6,10
6,7,49
7,8,42
8,9,16
9,10,29


In [8]:
conteo_edo = ubica_geo['CVE_ENT'].value_counts().reset_index()
conteo_edo.columns = ['edo', 'edo_reales']  # renombrar columnas

# 2️⃣ Asegurar que los tipos coincidan para poder unir
resumen_munic['edo'] = resumen_munic['edo'].astype(int)
conteo_edo['edo'] = conteo_edo['edo'].astype(int)

# 3️⃣ Unir los conteos a VIVIENDA
resumen_munic = resumen_munic.merge(conteo_edo, on='edo', how='left')

resumen_munic

,edo,munic_ENIGH,edo_reales
0,1,10,11
1,2,6,7
2,3,5,5
3,4,11,13
4,5,29,38
5,6,10,10
6,7,49,124
7,8,42,67
8,9,16,16
9,10,29,39


In [9]:
resumen_munic['munic_faltantes'] = resumen_munic['edo_reales'] - resumen_munic['munic_ENIGH'] 
resumen_munic

,edo,munic_ENIGH,edo_reales,munic_faltantes
0,1,10,11,1
1,2,6,7,1
2,3,5,5,0
3,4,11,13,2
4,5,29,38,9
5,6,10,10,0
6,7,49,124,75
7,8,42,67,25
8,9,16,16,0
9,10,29,39,10


In [10]:
# 1️⃣ Seleccionar columnas relevantes de ubica_geo
edo_nombre = ubica_geo[['CVE_ENT', 'NOM_ENT']]

# 2️⃣ Mantener solo el primer nombre de cada CVE_ENT
edo_nombre_unico = edo_nombre.drop_duplicates(subset='CVE_ENT')

# 3️⃣ Asegurarse que CVE_ENT sea numérico para unir con resumen_munic
edo_nombre_unico['CVE_ENT'] = edo_nombre_unico['CVE_ENT'].astype(int)
resumen_munic['edo'] = resumen_munic['edo'].astype(int)

# 4️⃣ Hacer merge para añadir la columna NOM_ENT
resumen_munic = resumen_munic.merge(
    edo_nombre_unico,
    left_on='edo',
    right_on='CVE_ENT',
    how='left'
)

# 5️⃣ Opcional: eliminar la columna CVE_ENT que quedó duplicada
resumen_munic = resumen_munic.drop(columns=['CVE_ENT'])

# 6️⃣ Revisar resultado
print(resumen_munic.head())


   edo  munic_ENIGH  edo_reales  munic_faltantes               NOM_ENT
0    1           10          11                1        Aguascalientes
1    2            6           7                1       Baja California
2    3            5           5                0   Baja California Sur
3    4           11          13                2              Campeche
4    5           29          38                9  Coahuila de Zaragoza


C:\Users\roele\AppData\Local\Temp\ipykernel_28016\3746256913.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  edo_nombre_unico['CVE_ENT'] = edo_nombre_unico['CVE_ENT'].astype(int)


In [11]:
resumen_munic.sort_values(by='munic_faltantes', ascending=True)

,edo,munic_ENIGH,edo_reales,munic_faltantes,NOM_ENT
2,3,5,5,0,Baja California Sur
5,6,10,10,0,Colima
8,9,16,16,0,Ciudad de México
21,22,18,18,0,Querétaro
0,1,10,11,1,Aguascalientes
1,2,6,7,1,Baja California
22,23,10,11,1,Quintana Roo
3,4,11,13,2,Campeche
26,27,15,17,2,Tabasco
24,25,17,20,3,Sinaloa


In [12]:
resumen_munic['edo_reales'].sum()

np.int64(2478)

Tengo que decidir entre asingar valores a los municipios de Oaxaca a partir de los datos de los municipios colindantes o agarrar al que tenga mas municipios en ENIGH y menos municipios faltantes 